**Dependencies Required**

In [4]:
!pip install -q langchain langchain-text-splitters langchain-chroma langchain-google-genai python-docx


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**Adding a API Key**

In [ ]:
import os
## from google.colab import files

## os.environ["GOOGLE_API_KEY"] = "..."  # Replace with your actual Google API key

ModuleNotFoundError: No module named 'google.colab'

**Now we need to read the Document**

Since this was a word document we need to import document from docx library to open and read it.

In [ ]:
from docx import Document

doc = Document("2. AI.docx")
text = ""

for para in doc.paragraphs:
    text += para.text + "\n"

print(text[:500]) # Quick peek at the text

AI - Artificial Intelligence
What is AI:
Artificial Intelligence (AI) is a branch of computer science that enables machines and computers to simulate human intelligence, allowing them to learn, reason, solve problems, and perform tasks that typically require human cognition like understanding language or recognizing patterns.
History of AI:
During WW2:
Sir Alan tuning: Mathematician: he thought can machine think. He broke the code and helped Britain: Machine, Cracks the code and tells what the m


**Split Into Chunks**

We need to store it into chunks because rag doesn't store the huge document directly.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)
print("Number of chunks:", len(chunks))

Number of chunks: 7


**Create Embedding**

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

**Store In ChomaDB**

In [ ]:
from langchain_chroma import Chroma

vector_db = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings
)

## NOW NOTES ARE INDEXED

**Create Retriever**

In [ ]:
# Give me only the 2 most relevant chunks
retriever = vector_db.as_retriever(
    search_kwargs={"k": 2}
)

**Connect Gemini**

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Explicitly specifying the model without the 'models/' prefix
# forces LangChain's latest routing to request the correct public tier
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

**RAG PROMPT**

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """
Answer the question using ONLY the context below.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate.from_template(template)

**Asking Question Function**

In [ ]:
import time
from langchain_google_genai.chat_models import ChatGoogleGenerativeAIError

def ask_notes(question):
    # Fetch the chunks
    docs = retriever.invoke(question)

    # Glue them together
    context = "\n\n".join([doc.page_content for doc in docs])

    # Format the prompt
    final_prompt = prompt.format(
        context=context,
        question=question
    )

    for attempt in range(2):
        try:
            response = llm.invoke(final_prompt)
            return response.content
        except ChatGoogleGenerativeAIError as e:
            if "RESOURCE_EXHAUSTED" in str(e) and attempt == 0:
                print("Pausing for 30 seconds to refresh quota...")
                time.sleep(30)
            else:
                raise e

**TEST!!**

In [ ]:
print(
    ask_notes("Who wrote these notes?")
)

Nikhil Srivastava
